In [1]:
import os
import openai
from huggingface_hub import InferenceClient
api_key = os.getenv("4061_API_KEY")
llama_key = os.getenv("NVIDIA_API_KEY")

C:\Users\Kyle\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [201]:
messages = [
        {"role":"system", "content":"You are a legal assistant, and your job is to find relevant case law and advice for writing legal briefs and motions."},
        {"role":"user", "content":"Give me an overview of medical malpractice cases involving endoscopic retrograde cholangiopancreatography (ERCP) in the state of Illinois. Cite ERCP-specific case law."},
        #{"role":"user", "content": "Give me an overview of Illinois state legislation and cases involving police."},
        #{"role":"user", "content":"Give me an overview of federal social security legislation and cases."}    
    ]
gpt_client = openai.OpenAI(api_key = api_key)
response = gpt_client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = messages
)

llama_client = openai.OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = llama_key
)


In [202]:
#gather the (potentially hallucinated) response
ans = response.choices[0].message.content
print(ans)

Medical malpractice cases involving endoscopic retrograde cholangiopancreatography (ERCP) in Illinois can be complex, typically revolving around issues of professional negligence, informed consent, and the standard of care expected from physicians performing this procedure.

### Overview of ERCP and Medical Malpractice Standards

ERCP is a specialized procedure primarily used for diagnosing and treating conditions related to the bile ducts and pancreas. Given the technical nature of ERCP, complications can arise, including but not limited to:

- Pancreatitis
- Infection
- Bowel perforation
- Bleeding
- Incomplete duct clearance

In Illinois, a medical malpractice claim requires the plaintiff to establish the following elements:

1. **Duty**: The healthcare provider owed a duty to the patient.
2. **Breach**: The provider breached that duty by failing to adhere to the accepted standard of care.
3. **Causation**: The breach of duty caused harm to the patient.
4. **Damages**: The patient s

In [203]:
#general one-shot self reflection
response2 = gpt_client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {"role":"system", "content":"You are a fact-checker. Determine whether the following case law exists or not. Return the input breif with all incorrect cases corrected or removed. "},
        {"role":"user", "content": ans}])    

In [204]:
print(response2.choices[0].message.content)

Medical malpractice cases involving endoscopic retrograde cholangiopancreatography (ERCP) in Illinois can be complex, typically revolving around issues of professional negligence, informed consent, and the standard of care expected from physicians performing this procedure.

### Overview of ERCP and Medical Malpractice Standards

ERCP is a specialized procedure primarily used for diagnosing and treating conditions related to the bile ducts and pancreas. Given the technical nature of ERCP, complications can arise, including but not limited to:

- Pancreatitis
- Infection
- Bowel perforation
- Bleeding
- Incomplete duct clearance

In Illinois, a medical malpractice claim requires the plaintiff to establish the following elements:

1. **Duty**: The healthcare provider owed a duty to the patient.
2. **Breach**: The provider breached that duty by failing to adhere to the accepted standard of care.
3. **Causation**: The breach of duty caused harm to the patient.
4. **Damages**: The patient s

In [205]:
#find-fix-verify pipeline - multiple models
#find step
num_it = 0
resolved = False
unverified_text = ans

while not resolved and num_it < 5:
    #want the most objective, determinstic results for finding the hallucinations because it is not a creative task
    find_step = gpt_client.chat.completions.create(
        model="gpt-4o",
        temperature = 0,
        messages = [
            {"role" : "system", "content" : "You are a professional legal fact-checker. Do not change, delete, or summarize any content. Only identify **legal case citations** and append `[FIX]` directly after each case name. Be diligent about identifying all potential references to court cases. Output must be identical to the input, except for `[FIX]` annotations. Do not justify any answers, only add [FIX] tags. Do not modify wording, spacing, or structure."},
            {"role" : "user", "content" : unverified_text}]
    )
    found = find_step.choices[0].message.content
    
    #fix step - default temperature is okay here, it is the more creative portion of the pipeline
    fix_step = llama_client.chat.completions.create(
        model = "meta/llama-3.1-8b-instruct",
        messages = [
            {"role" : "system", "content" : "You are a legal expert responsible for validating case citations. Each `[FIX]` tag indicates a citation that may be hallucinated. For each: If the citation is hallucinated, delete it. If it is real, keep it as-is. If you're unsure, delete it. Keep the rest of the input **exactly** the same. Do not add any tags. Do not rewrite, paraphrase, or delete anything except citations labeled `[FIX]`. Do not justify any answers, only delete or leave in place. If the case is found to be false, actually remove it from the brief. Final output: cleaned-up legal text with false citations removed." },
            {"role" : "user", "content" : found}]
    )
    fixed = fix_step.choices[0].message.content
    
    #verify step
    verify_step = llama_client.chat.completions.create(
        model="google/gemma-3-1b-it",
        temperature = 0,
        messages = [
            {"role" : "system", "content" : "You are a professional fact-checker. Respond in one word: TRUE or FALSE. Ensure every legal case cited is factual. Assume that the user input is likely to contain errors. If you find one case that has a chance of being incorrect, return FALSE. It is better to be cautious and return FALSE. Only return TRUE if you are 100% confident in all cases present."},
            {"role" : "user", "content" : fixed}]
    )
    verified = verify_step.choices[0].message.content
    print(verified)
    resolved = verified == "TRUE"
    num_it += 1
    unverified_text = fixed
print(num_it)
final_ans = fixed

TRUE
1


In [206]:
print(final_ans)

Medical malpractice cases involving endoscopic retrograde cholangiopancreatography (ERCP) in Illinois can be complex, typically revolving around issues of professional negligence, informed consent, and the standard of care expected from physicians performing this procedure.

### Overview of ERCP and Medical Malpractice Standards

ERCP is a specialized procedure primarily used for diagnosing and treating conditions related to the bile ducts and pancreas. Given the technical nature of ERCP, complications can arise, including but not limited to:

- Pancreatitis
- Infection
- Bowel perforation
- Bleeding
- Incomplete duct clearance

In Illinois, a medical malpractice claim requires the plaintiff to establish the following elements:

1. **Duty**: The healthcare provider owed a duty to the patient.
2. **Breach**: The provider breached that duty by failing to adhere to the accepted standard of care.
3. **Causation**: The breach of duty caused harm to the patient.
4. **Damages**: The patient s

In [207]:
#find-fix-verify pipeline - all openai
#find step
num_it = 0
resolved = False
unverified_text = ans

while not resolved and num_it < 5:
    #want the most objective, determinstic results for finding the hallucinations because it is not a creative task
    find_step = gpt_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature = 0,
        messages = [
            {"role" : "system", "content" : "You are a professional legal fact-checker. Do not change, delete, or summarize any content. Only identify **legal case citations** and append `[FIX]` directly after each case name. Be diligent about identifying all potential references to court cases. Output must be identical to the input, except for `[FIX]` annotations. Do not justify any answers, only add [FIX] tags. Do not modify wording, spacing, or structure."},
            {"role" : "user", "content" : unverified_text}]
    )
    found = find_step.choices[0].message.content
    
    #fix step - default temperature is okay here, it is the more creative portion of the pipeline
    fix_step = gpt_client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [
            {"role" : "system", "content" : "You are a legal expert responsible for validating case citations. Each `[FIX]` tag indicates a citation that may be hallucinated. For each: If the citation is hallucinated, delete it. If it is real, keep it as-is. If you're unsure, delete it. Keep the rest of the input **exactly** the same. Do not add any tags. Do not rewrite, paraphrase, or delete anything except citations labeled `[FIX]`. Do not justify any answers, only delete or leave in place. If the case is found to be false, actually remove it from the brief. Final output: cleaned-up legal text with false citations removed." },
            {"role" : "user", "content" : find_step.choices[0].message.content}]
    )
    fixed = fix_step.choices[0].message.content
    
    #verify step
    verify_step = gpt_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature = 0,
        messages = [
            {"role" : "system", "content" : "You are a professional fact-checker. Respond in one word: TRUE or FALSE. Ensure every legal case cited is factual. Assume that the user input is likely to contain errors. If you find one case that has a chance of being incorrect, return FALSE. It is better to be cautious and return FALSE. Only return TRUE if you are 100% confident in all cases present."},
            {"role" : "user", "content" : fixed}]
    )
    verified = verify_step.choices[0].message.content
    print(verified)
    resolved = verified == "TRUE"
    num_it += 1
    unverified_text = fixed
print(num_it)
final_ans = fixed

FALSE
TRUE
2


In [208]:
print(final_ans)

Medical malpractice cases involving endoscopic retrograde cholangiopancreatography (ERCP) in Illinois can be complex, typically revolving around issues of professional negligence, informed consent, and the standard of care expected from physicians performing this procedure.

### Overview of ERCP and Medical Malpractice Standards

ERCP is a specialized procedure primarily used for diagnosing and treating conditions related to the bile ducts and pancreas. Given the technical nature of ERCP, complications can arise, including but not limited to:

- Pancreatitis
- Infection
- Bowel perforation
- Bleeding
- Incomplete duct clearance

In Illinois, a medical malpractice claim requires the plaintiff to establish the following elements:

1. **Duty**: The healthcare provider owed a duty to the patient.
2. **Breach**: The provider breached that duty by failing to adhere to the accepted standard of care.
3. **Causation**: The breach of duty caused harm to the patient.
4. **Damages**: The patient s